In [159]:
from scipy.sparse import csc_array, coo_array
import fast_matrix_market as fmm
from os.path import join
import numpy as np
import matplotlib.pyplot as plt

def mmread(source,sorted=False):
    mmdata = fmm.mmread(source)
    data = mmdata.data
    rows = mmdata.row
    cols = mmdata.col
    if sorted:
        indexs = [(x,y) for x,y in zip(rows,cols)]
        eindexs = [(i,x) for i,x in enumerate(indexs)]
        sortedindexs = sorted(eindexs, key=lambda x: (x[1][1], x[1][0]))
        sfindexs = [x[0] for x in sortedindexs]
        data = data[sfindexs]
        rows = rows[sfindexs]
        cols = cols[sfindexs]
        print("in sort", )
    print("in mmread cols rows size ", len(rows))
    return coo_array((data, (rows,cols)), shape=mmdata.shape).tocsc()

def sortcsc(spcsc):
    coo = spcsc.tocoo()
    data = coo.data
    rows = coo.row
    cols = coo.col
    print("in sort csc size ",len(data))
    indexs = [(x,y) for x,y in zip(rows,cols)]
    eindexs = [(i,x) for i,x in enumerate(indexs)]
    sortedindexs = sorted(eindexs, key=lambda x: (x[1][1], x[1][0]))
    sfindexs = [x[0] for x in sortedindexs]
    data = data[sfindexs]
    rows = rows[sfindexs]
    cols = cols[sfindexs]
    print(len(sfindexs))
    return coo_array((data, (rows,cols)), shape=coo.shape).tocsc()

def checkeq(m1,m2):
    m1 = sortcsc(m1)
    m2 = sortcsc(m2)
    ret = 0
    ret += len(m1.indices) != len(m2.indices)
    if ret != 0:
        print("index length not equal!")
        return False
    ret += (sum(m1.indices != m2.indices))
    ret += (sum(m1.indptr != m2.indptr))
    if ret != 0:
        print("index are wrong!")
        return False
    if ret == 0 and np.allclose(m1.data, m2.data):
        print("equal!")
        return True
    else:
        print("data are wrong!")
    return False


Rm = 45101
Rn = 4086
mpisize = 16

# SpGEMM 1D algorithm
def getblocksize(Rm,Rn,mpisize):
    rowblocksizevec = np.zeros(mpisize,dtype=np.int64)
    colblocksizevec = np.zeros(mpisize,dtype=np.int64)
    rowblockprefix  = np.zeros(mpisize,dtype=np.int64)
    colblockprefix  = np.zeros(mpisize,dtype=np.int64)
    rbs = Rm // mpisize
    cbs = Rn // mpisize
    for i in range(15):
        rowblocksizevec[i] = rbs
        colblocksizevec[i] = cbs
    rowblocksizevec[-1] = (Rm - rbs * (mpisize-1))
    colblocksizevec[-1] = (Rn - cbs * (mpisize-1))
    for i in range(1, mpisize):
        rowblockprefix[i] = rowblockprefix[i-1] + rowblocksizevec[i-1] 
        colblockprefix[i] = colblockprefix[i-1] + colblocksizevec[i-1]
    return rowblocksizevec, rowblockprefix, colblocksizevec, colblockprefix

rowblocksizevec, rowblockprefix, colblocksizevec, colblockprefix = getblocksize(Rm, Rn, mpisize)
basepath = "/pscratch/sd/y/yuxihong/graphclustering/combblascorrect"
# print(np.sum(colblocksizevec))
# print(colblocksizevec)
# print(rowblocksizevec)
# print(len(colblockprefix))
# print(colblockprefix)

In [137]:
browlist = []
for target in range(16):
    brow0_data = []
    brow0_rows = []
    brow0_cols = []
    for i in range(16):  
        tmpspmat = mmread( join(basepath, f"spSeqDER{i}.mtx") )
        tmpcnt = 0
        for colid in range( len(tmpspmat.indptr)-1 ):
            startidx = tmpspmat.indptr[colid]
            endidx = tmpspmat.indptr[colid+1]
            for index in range(startidx,endidx):
                rowid = tmpspmat.indices[index]
                value = tmpspmat.data[index]
                if rowid >= Rm:
                    print("rowid wrong, ", rowid)
                if colid >= colblocksizevec[i]:
                    print(i, " colid larger ", colid, ", ", colblocksizevec[i])
                if target != 15 and rowid < rowblockprefix[target+1] and rowid >= rowblockprefix[target]:
                    brow0_rows.append(rowid - rowblockprefix[target])
                    brow0_cols.append(colid + colblockprefix[i])
                    brow0_data.append(value)
                    tmpcnt += 1
                elif target == 15 and rowid >= rowblockprefix[target]:
                    brow0_rows.append(rowid - rowblockprefix[target])
                    brow0_cols.append(colid + colblockprefix[i])
                    brow0_data.append(value)
                    tmpcnt += 1
    # print(f"rank {target} length {len(brow0_data)}")
    brow0seq = sortcsc( coo_array( (brow0_data,(brow0_rows,brow0_cols)), shape=(rowblocksizevec[target],Rn)) )
    browlist.append(brow0seq)
    browsDER0 = mmread(join(basepath,f"browsDER{target}.mtx"))
    checkeq(brow0seq,browsDER0)
    # print(ret)
# print(len(brow0_data))
# brow0seq = sortcsc( coo_array( (brow0_data,(brow0_rows,brow0_cols)), shape=(rowblocksizevec[0],Rn)) )

equal!
equal!
equal!
equal!
equal!
equal!
equal!
equal!
equal!
equal!
equal!
equal!
equal!
equal!
equal!
equal!


In [160]:
OP_A = sortcsc( mmread(join(basepath,"OP_A.mtx")) )
OP_B = sortcsc( mmread(join(basepath,"OP_B.mtx")) )
# OP_C = mmread(join(basepath,"OP_C.mtx"))
resop = []
for i in range(16):
    if i == 15:
        subA = OP_A[:,rowblockprefix[i]:]
        subB = OP_B[rowblockprefix[i]:,:]
    else:
        subA = OP_A[:,rowblockprefix[i]:rowblockprefix[i+1]]
        subB = OP_B[rowblockprefix[i]:rowblockprefix[i+1],:]
    # tmpA = sortcsc( mmread(join(basepath,f"ADER{i}.mtx")) )
    # tmpB = sortcsc( mmread(join(basepath,f"browsDER{i}.mtx")) )
    # checkeq(tmpA, subA)
    # checkeq(tmpB, subB)
    # print("op !")
    resop.append(subA @ subB)

in mmread cols rows size  2505780
in sort csc size  2505780
2505780
in mmread cols rows size  45101
in sort csc size  45101
45101


In [161]:
for i in range(0,16):
    PartC = sortcsc(mmread(join(basepath, f"PartC{i}.mtx")))
    checkeq(resop[i],PartC)

in mmread cols rows size  32505
in sort csc size  32505
32505
in sort csc size  32505
32505
in sort csc size  32505
32505
equal!
in mmread cols rows size  36038
in sort csc size  36038
36038
in sort csc size  36038
36038
in sort csc size  36038
36038
equal!
in mmread cols rows size  37658
in sort csc size  37658
37658
in sort csc size  37658
37658
in sort csc size  37658
37658
equal!
in mmread cols rows size  34745
in sort csc size  34745
34745
in sort csc size  34745
34745
in sort csc size  34745
34745
equal!
in mmread cols rows size  35990
in sort csc size  35990
35990
in sort csc size  35990
35990
in sort csc size  35990
35990
equal!
in mmread cols rows size  36715
in sort csc size  36715
36715
in sort csc size  36715
36715
in sort csc size  36715
36715
equal!
in mmread cols rows size  38743
in sort csc size  38743
38743
in sort csc size  38743
38743
in sort csc size  38743
38743
equal!
in mmread cols rows size  37021
in sort csc size  37021
37021
in sort csc size  37021
37021
in so

In [162]:
for i in range(1,16):
    resop[0] += resop[i]

In [163]:
correct = OP_A @ OP_B
checkeq(resop[0],correct)
tot = 0
for i in range(16):
    if i == 15:
        subC = correct[:,colblockprefix[i]:]
    else:
        subC = correct[:,colblockprefix[i]:colblockprefix[i+1]]
    print(f"rank {i} subC size ", subC.nnz)
    tot += subC.nnz
print(tot)

in sort csc size  248044
248044
in sort csc size  248044
248044
equal!
rank 0 subC size  13530
rank 1 subC size  15797
rank 2 subC size  11466
rank 3 subC size  15050
rank 4 subC size  14650
rank 5 subC size  17222
rank 6 subC size  13521
rank 7 subC size  17210
rank 8 subC size  14123
rank 9 subC size  17693
rank 10 subC size  16304
rank 11 subC size  15572
rank 12 subC size  15792
rank 13 subC size  16020
rank 14 subC size  17195
rank 15 subC size  16899
248044


In [164]:
OP_C = sortcsc(mmread(join(basepath,"OP_C.mtx")))
mmdata = fmm.mmread(join(basepath,"OP_C.mtx"))
# print(len(mmdata.data))
# print(OP_C.nnz)
# checkeq(resop[0],OP_C)

data = mmdata.data
rows = mmdata.row
cols = mmdata.col
indexs = [(x,y) for x,y in zip(rows,cols)]
eindexs = [(i,x) for i,x in enumerate(indexs)]
sortedindexs = sorted(eindexs, key=lambda x: (x[1][1], x[1][0]))
sfindexs = [x[0] for x in sortedindexs]
data = data[sfindexs]
rows = rows[sfindexs]
cols = cols[sfindexs]
print("in sort", )

in mmread cols rows size  270845
in sort csc size  270845
270845
in sort


array([   0,    3,    4, ..., 4091, 4092, 4093], dtype=int32)

In [158]:
cols

array([  0,   0,   0, ..., 266, 267, 268], dtype=int32)

In [145]:
len(correct.indices)

239158

In [146]:
len(OP_C.indices)

160207

In [134]:
177247 * 2

354494